# 01 — Data Exploration

This notebook explores the three benchmark datasets used in the GT-ADF paper:
- CICIDS2017-SafeML
- UNSW-NB15
- ToN-IoT

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_style('whitegrid')

In [ ]:
# ── CICIDS2017 ─────────────────────────────────────────────────────
# Update path to your downloaded dataset
CICIDS_PATH = '../data/raw/CICIDS2017/'
import os
files = [f for f in os.listdir(CICIDS_PATH) if f.endswith('.csv')]
dfs = [pd.read_csv(os.path.join(CICIDS_PATH, f), low_memory=False) for f in files[:2]]
df_cic = pd.concat(dfs, ignore_index=True)
df_cic.columns = df_cic.columns.str.strip()
print('Shape:', df_cic.shape)
df_cic['Label'].value_counts()

In [ ]:
# Label distribution
fig, ax = plt.subplots(figsize=(12, 4))
df_cic['Label'].value_counts().plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('CICIDS2017 — Attack Type Distribution')
ax.set_xlabel('Attack Type')
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Feature correlation heatmap (sample of 20 features)
numeric_df = df_cic.select_dtypes(include='number').dropna(axis=1)
sample_cols = numeric_df.columns[:20]
corr = numeric_df[sample_cols].corr()
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, ax=ax, cmap='coolwarm', center=0, annot=False)
ax.set_title('Feature Correlation Matrix (first 20 features)')
plt.tight_layout()
plt.show()

## Preprocessing pipeline

The `Preprocessor` class handles:
1. Removal of constant features (VarianceThreshold)
2. Removal of highly correlated features (threshold=0.98)
3. NaN/Inf handling
4. Min-Max normalization to [0, 1]

In [ ]:
from src.data.preprocessor import Preprocessor
prep = Preprocessor()
df_cic['label_enc'] = df_cic['Label'].map({
    'BENIGN': 0, 'Bot': 1, 'DDoS': 2, 'DoS GoldenEye': 3,
    'DoS Hulk': 4, 'DoS Slowhttptest': 5, 'DoS slowloris': 6,
    'FTP-Patator': 7, 'Heartbleed': 8, 'Infiltration': 9,
    'PortScan': 10, 'SSH-Patator': 11,
    'Web Attack \u2013 Brute Force': 12,
    'Web Attack \u2013 Sql Injection': 13,
    'Web Attack \u2013 XSS': 14,
}).fillna(-1).astype(int)
df_valid = df_cic[df_cic['label_enc'] >= 0]
X, y = prep.fit_transform(df_valid, label_col='label_enc')
print(f'Features after preprocessing: {X.shape[1]}')
print(f'Value range: [{X.values.min():.3f}, {X.values.max():.3f}]')